# Semana 10 · ¿Le creo al modelo?

**Principios de Inteligencia Artificial · CORHUILA**

**No tienes que escribir código.** Ejecuta las celdas en orden, de arriba abajo: haz clic en una celda y presiona **Shift + Enter**, o el botón ▶ que aparece a su izquierda.

Cuando haya algo que cambiar, estará marcado con **⬅️ CAMBIA ESTE NÚMERO**. No toques nada más: si algo se daña, en el menú *Entorno de ejecución → Reiniciar y ejecutar todo* vuelve a dejarlo como estaba.

Vas a entrenar un árbol de decisión y a medir su acierto de dos maneras: con los datos que ya vio y con datos que nunca vio. Es la misma **prueba de la memoria** que hiciste en Orange con *Test and Score*.

In [ ]:
# 1. Traer las herramientas (se ejecuta una sola vez)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
print("Listo: herramientas cargadas.")

## 1. Los datos

Son 300 estudiantes **inventados**. De cada uno sabemos cuántas horas estudia a la semana, qué porcentaje de clases asiste y cuántas tareas entregó de 10. La etiqueta es si **aprobó** o no.

Como en la vida real, los datos tienen algo de desorden: hay estudiantes que estudiaron mucho y reprobaron, y otros al revés. Ese desorden es justamente lo que un modelo **no** debería aprenderse de memoria.

In [ ]:
# 2. Crear los 300 estudiantes (siempre salen los mismos)
rng = np.random.default_rng(42)
n = 300
estudiantes = pd.DataFrame({
    "horas": rng.uniform(0, 10, n).round(1),
    "asistencia": rng.uniform(40, 100, n).round(0),
    "tareas": rng.integers(0, 11, n),
})
puntaje = 0.35*estudiantes.horas + 0.04*estudiantes.asistencia + 0.25*estudiantes.tareas
aprobo = puntaje > np.median(puntaje)
desorden = rng.random(n) < 0.10          # 10% de casos que no siguen la regla
estudiantes["aprobó"] = np.where(desorden, ~aprobo, aprobo)
estudiantes["aprobó"] = estudiantes["aprobó"].map({True: "sí", False: "no"})
estudiantes.head()

## 2. Apartar los datos de prueba

Antes de entrenar, separamos los datos: el **70%** para que el árbol aprenda y el **30%** lo guardamos bajo llave. El árbol **nunca** verá esos datos mientras aprende: los usaremos solo para examinarlo.

In [ ]:
# 3. Separar entrenamiento (70%) y prueba (30%)
X = estudiantes[["horas", "asistencia", "tareas"]]
y = estudiantes["aprobó"]
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y, test_size=0.3, random_state=0)
print("Para aprender:", len(X_entrena), "estudiantes")
print("Para probar:  ", len(X_prueba), "estudiantes")

## 3. Entrenar el árbol y examinarlo

La **profundidad** es el número máximo de preguntas seguidas que el árbol puede hacer antes de decidir. Un árbol muy profundo puede hacer tantas preguntas que termina memorizando a cada estudiante, desorden incluido.

Empieza con **profundidad = 20** y ejecuta la celda.

In [ ]:
# 4. Entrenar y medir el acierto de las dos maneras
profundidad = 20   # ⬅️ CAMBIA ESTE NÚMERO: prueba 20, 5, 3 y 2

arbol = DecisionTreeClassifier(max_depth=profundidad, random_state=0)
arbol.fit(X_entrena, y_entrena)

con_lo_que_vio = accuracy_score(y_entrena, arbol.predict(X_entrena))
con_lo_nuevo = accuracy_score(y_prueba, arbol.predict(X_prueba))
print(f"Profundidad {profundidad}")
print(f"  Exactitud con los datos que YA VIO:   {con_lo_que_vio:.0%}")
print(f"  Exactitud con datos que NUNCA VIO:    {con_lo_nuevo:.0%}")

## 4. ¿En qué se equivoca?

La matriz de confusión, igual que el bloque *Confusion Matrix* de Orange. Se calcula solo con los datos de **prueba**, que son los que dicen la verdad sobre el modelo.

In [ ]:
# 5. Matriz de confusion con los datos de prueba
ConfusionMatrixDisplay.from_estimator(arbol, X_prueba, y_prueba, cmap="Greens")
plt.title(f"Árbol de profundidad {profundidad} · datos de prueba")
plt.show()

## 5. Lo que tienes que responder

1. Con **profundidad = 20**, ¿cuánto acierta con los datos que ya vio? ¿Y con los que nunca vio? ¿Cuál de los dos números le crees?
2. Prueba con **5, 3 y 2**. Anota los dos números en cada caso. ¿Con cuál profundidad acierta más con los datos **nuevos**?
3. ¿Por qué el árbol más profundo acierta casi todo con lo que ya vio, pero no con lo nuevo?
4. En la matriz de confusión: ¿cuántos estudiantes que **sí** aprobaron predijo el árbol como reprobados?